# PRAGYAAN – Units 1 and 2
Machine Learning project (21CSC305P). Data: English subset of the Kaggle Multilingual Customer Support Tickets dataset.

In [ ]:
# Upload tickets.parquet and emb.npy (from the zip) to the Colab session first, then run all cells.
!pip -q install xgboost hmmlearn pyarrow
import os; os.makedirs('/content/report_work/fig', exist_ok=True)
!mv -f /content/tickets.parquet /content/emb.npy /content/report_work/ 2>/dev/null; ls /content/report_work

In [ ]:
%%writefile /content/common.py
"""Shared data loading, split and plotting style for the PRAGYAAN report experiments."""
import re
import sys
import time

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

import re as _re
_PH = _re.compile(r"<(?:tel_num|email|url|account_num|order_num|ip|date|name|address|password|username|phone|amount|id)[^>]*>", _re.I)


def clean_text(text, strict=False):
    out = _re.sub(r"\s+", " ", _PH.sub(" ", text or "")).strip()
    return out.lower() if strict else out

W = "/content/report_work"
FIG = f"{W}/fig"
LABELS = ["high", "medium", "low"]
SEED = 42

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.bbox": "tight",
})
C = {"high": "#B23A48", "medium": "#E0A458", "low": "#4E8098", "ink": "#2B2D42", "grey": "#8D99AE"}


def load():
    df = pd.read_parquet(f"{W}/tickets.parquet")
    for c in ["text", "priority", "type", "queue", "answer"]:
        df[c] = np.array(df[c].tolist(), dtype=object)
    emb = np.load(f"{W}/emb.npy")
    df["clean"] = [clean_text(t, strict=True) for t in df["text"]]
    df["words"] = df["text"].str.split().str.len()
    return df, emb


def split(df):
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=0.2, stratify=df["priority"], random_state=SEED)
    return tr, te


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words="english", min_df=2)


def evaluate(name, model, Xtr, ytr, Xte, yte, proba=True):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_s = time.perf_counter() - t0
    pred = model.predict(Xte)
    p, r, f, s = precision_recall_fscore_support(yte, pred, labels=LABELS, zero_division=0)
    out = {"model": name, "accuracy": accuracy_score(yte, pred), "macro_f1": f1_score(yte, pred, average="macro"),
           "weighted_f1": f1_score(yte, pred, average="weighted"), "fit_seconds": fit_s,
           "per_class": {lab: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                         for i, lab in enumerate(LABELS)}}
    if proba and hasattr(model, "predict_proba"):
        try:
            P = model.predict_proba(Xte)
            out["roc_auc_macro"] = roc_auc_score(yte, P, multi_class="ovr", average="macro", labels=list(model.classes_))
        except Exception:
            pass
    elif hasattr(model, "decision_function"):
        D = model.decision_function(Xte)
        out["roc_auc_macro"] = _auc_from_scores(yte, D, list(model.classes_))
    out["pred"] = list(pred)
    return out


def _auc_from_scores(y, D, classes):
    from scipy.special import softmax
    return roc_auc_score(y, softmax(D, axis=1), multi_class="ovr", average="macro", labels=classes)


def words_re(s):
    return re.findall(r"[a-z]+", s.lower())


In [ ]:
"""Units 1 and 2: EDA, TF-IDF features, Logistic Regression, Naive Bayes, Linear SVM, k-NN and MLP."""
import json

import numpy as np
import pandas as pd
import sys; sys.path.insert(0, '/content')
from common import FIG, LABELS, SEED, C, W, evaluate, load, plt, split, tfidf
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

df, emb = load()
tr, te = split(df)
R = {"n_total": len(df), "n_train": len(tr), "n_test": len(te)}

# ------------------------------------------------------------------ EDA
R["priority_counts"] = df["priority"].value_counts().to_dict()
R["type_counts"] = df["type"].value_counts().to_dict()
R["queue_counts"] = df["queue"].value_counts().to_dict()
R["words"] = df["words"].describe().round(1).to_dict()
R["words_by_priority"] = df.groupby("priority")["words"].median().to_dict()
R["answer_words_median"] = float(df["answer"].str.split().str.len().median())
R["dup_text"] = int(df["text"].duplicated().sum())
R["priority_by_type"] = (pd.crosstab(df["type"], df["priority"], normalize="index") * 100).round(1).to_dict()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 1.2, 2.4]})
pc = df["priority"].value_counts().reindex(LABELS)
ax[0].bar(pc.index, pc.values, color=[C[x] for x in pc.index])
ax[0].set_title("(a) Priority"); ax[0].set_ylabel("Tickets")
for i, v in enumerate(pc.values):
    ax[0].text(i, v + 60, f"{v}\n({v / len(df):.1%})", ha="center", fontsize=9)
ax[0].set_ylim(0, pc.max() * 1.25)
tc = df["type"].value_counts()
ax[1].bar(tc.index, tc.values, color=C["grey"])
ax[1].set_title("(b) Ticket type")
for i, v in enumerate(tc.values):
    ax[1].text(i, v + 50, str(v), ha="center", fontsize=9)
qc = df["queue"].value_counts().sort_values()
ax[2].barh(qc.index, qc.values, color=C["ink"])
ax[2].set_title("(c) Support queue (department)")
for i, v in enumerate(qc.values):
    ax[2].text(v + 20, i, str(v), va="center", fontsize=9)
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_2_class_distribution.png"); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for lab in LABELS:
    ax[0].hist(df.loc[df.priority == lab, "words"], bins=40, alpha=0.55, label=lab, color=C[lab])
ax[0].set_xlabel("Words per ticket (subject + body)"); ax[0].set_ylabel("Tickets"); ax[0].legend(title="Priority")
ax[0].set_title("(a) Ticket length by priority")
ct = pd.crosstab(df["type"], df["priority"], normalize="index")[LABELS] * 100
im = ax[1].imshow(ct.values, cmap="Greys", vmin=0, vmax=60)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(LABELS); ax[1].set_yticks(range(len(ct))); ax[1].set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(3):
        v = ct.values[i, j]
        ax[1].text(j, i, f"{v:.1f}%", ha="center", va="center", color="white" if v > 35 else "black", fontsize=10)
ax[1].set_title("(b) Priority mix within each ticket type"); ax[1].set_xlabel("Priority")
ax[1].spines[:].set_visible(False)
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_3_length_and_type.png"); plt.close()

# ------------------------------------------------------------------ features
vec = tfidf()
Xtr = vec.fit_transform(df["clean"].iloc[tr]); Xte = vec.transform(df["clean"].iloc[te])
ytr = np.array(df["priority"].iloc[tr].tolist(), dtype=object); yte = np.array(df["priority"].iloc[te].tolist(), dtype=object)
R["tfidf_vocab"] = len(vec.vocabulary_)
R["tfidf_density"] = float(Xtr.nnz / (Xtr.shape[0] * Xtr.shape[1]))
terms = np.array(vec.get_feature_names_out())
top_terms = {}
for lab in LABELS:
    s, _ = chi2(Xtr, (ytr == lab).astype(int))
    s = np.nan_to_num(s)
    # keep terms more frequent in this class than elsewhere
    mean_in = np.asarray(Xtr[ytr == lab].mean(0)).ravel(); mean_out = np.asarray(Xtr[ytr != lab].mean(0)).ravel()
    s[mean_in <= mean_out] = 0
    top_terms[lab] = list(terms[np.argsort(-s)[:10]])
R["top_terms"] = top_terms

# ------------------------------------------------------------------ models
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
models = {}
# Unit 1
models["Logistic Regression"] = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=SEED)
models["Multinomial Naive Bayes"] = MultinomialNB(alpha=0.5)
models["Complement Naive Bayes"] = ComplementNB(alpha=0.5)
# Unit 2: tune C for SVM and k for k-NN with 5-fold CV on the training part
svm_cv = {}
for c in [0.05, 0.1, 0.5, 1.0]:
    svm_cv[c] = cross_val_score(LinearSVC(C=c, class_weight="balanced", random_state=SEED), Xtr, ytr, cv=cv,
                                scoring="f1_macro").mean()
best_c = max(svm_cv, key=svm_cv.get)
R["svm_cv"] = {str(k): v for k, v in svm_cv.items()}; R["svm_best_C"] = best_c
models["Linear SVM"] = LinearSVC(C=best_c, class_weight="balanced", random_state=SEED)
knn_cv = {}
for k in [5, 11, 21, 31, 51]:
    knn_cv[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance"), Xtr, ytr,
                                cv=cv, scoring="f1_macro").mean()
best_k = max(knn_cv, key=knn_cv.get)
R["knn_cv"] = {str(k): v for k, v in knn_cv.items()}; R["knn_best_k"] = best_k
models["k-NN (cosine)"] = KNeighborsClassifier(n_neighbors=best_k, metric="cosine", weights="distance")
models["MLP (128-64)"] = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=30, early_stopping=True, random_state=SEED)

results = []
for name, m in models.items():
    r = evaluate(name, m, Xtr, ytr, Xte, yte)
    r["unit"] = 1 if name in ("Logistic Regression", "Multinomial Naive Bayes", "Complement Naive Bayes") else 2
    results.append(r)
    print(f"{name:26s} acc={r['accuracy']:.4f} f1={r['macro_f1']:.4f} auc={r.get('roc_auc_macro', 0):.4f} t={r['fit_seconds']:.1f}s")

# embeddings + LR (same split) for the feature comparison
lr_emb = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
r = evaluate("Logistic Regression on MiniLM embeddings", lr_emb, emb[tr], ytr, emb[te], yte); r["unit"] = 2
results.append(r)
print(f"{r['model']:26s} acc={r['accuracy']:.4f} f1={r['macro_f1']:.4f}")

# 5-fold CV stability for the two linear models
R["cv_lr"] = cross_val_score(models["Logistic Regression"], Xtr, ytr, cv=cv, scoring="f1_macro").tolist()

# majority baseline
R["baseline_majority_acc"] = float((yte == "medium").mean())
R["baseline_majority_f1"] = float(f1_score(yte, np.full(len(yte), "medium"), average="macro"))

# confusion matrix of the best Unit 1-2 model
best = max([x for x in results if x["model"] != "Logistic Regression on MiniLM embeddings"], key=lambda x: x["macro_f1"])
cm = confusion_matrix(yte, best["pred"], labels=LABELS)
R["best_unit12"] = best["model"]; R["best_unit12_cm"] = cm.tolist()
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap="Greys")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() * 0.55 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(LABELS); ax.set_yticks(range(3)); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted priority"); ax.set_ylabel("True priority"); ax.spines[:].set_visible(False)
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_5_confusion_best.png"); plt.close()

# bar chart of macro-F1
fig, ax = plt.subplots(figsize=(8, 3.4))
order = sorted(results, key=lambda x: x["macro_f1"])
ax.barh([x["model"] for x in order], [x["macro_f1"] for x in order],
        color=[C["ink"] if x["unit"] == 2 else C["grey"] for x in order])
ax.axvline(R["baseline_majority_f1"], ls="--", color=C["high"], lw=1)
ax.text(R["baseline_majority_f1"] + 0.005, -0.6, "majority-class baseline", color=C["high"], fontsize=9)
for i, x in enumerate(order):
    ax.text(x["macro_f1"] + 0.004, i, f"{x['macro_f1']:.3f}", va="center", fontsize=9)
ax.set_xlabel("Macro-F1 on the held-out test split (1,907 tickets)"); ax.set_xlim(0, max(x["macro_f1"] for x in results) * 1.15)
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_4_model_comparison.png"); plt.close()

for x in results:
    x.pop("pred")
R["results"] = results
json.dump(R, open(f"{W}/unit12.json", "w"), indent=1, default=float)
print("best:", best["model"], "baseline acc", R["baseline_majority_acc"], "top terms", top_terms)
